# Build Performance Analysis

Deep analysis of build times, critical path, parallelism, and what-if scenarios. Identifies bottlenecks and simulates optimization opportunities.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("../../data/processed")
PROJECT_ROOT = Path("../..")
INTERMEDIATE_DIR = Path("../../data/intermediate")

from buildanalysis.loading import BuildDataset
ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
from buildanalysis.build import CriticalPathResult, PoolConfig, compute_critical_path, simulate_build, validate_simulation, whatif_remove_edge, whatif_reduce_target_time
from buildanalysis.graph import build_dependency_graph, compute_centrality_metrics
from buildanalysis.simulation import rebuild_cost, expected_daily_cost, simulate_incremental_build, simulate_merge, simulate_split, replay_git_history

## 1. Build Time Overview

Total build time breakdown: compile, link, and codegen components.

In [ ]:
target_metrics = ds.target_metrics

compile_time_ms = target_metrics['compile_time_sum_ms'].sum()
link_time_ms = target_metrics['link_time_ms'].sum()
codegen_time_ms = target_metrics['codegen_time_ms'].fillna(0).sum()

total_time_ms = compile_time_ms + link_time_ms + codegen_time_ms

print(f"Build Time Summary:")
print(f"  Compile time: {compile_time_ms:,.0f} ms ({compile_time_ms/total_time_ms*100:.1f}%)")
print(f"  Link time:    {link_time_ms:,.0f} ms ({link_time_ms/total_time_ms*100:.1f}%)")
print(f"  Codegen time: {codegen_time_ms:,.0f} ms ({codegen_time_ms/total_time_ms*100:.1f}%)")
print(f"  Total:        {total_time_ms:,.0f} ms ({total_time_ms/1000:.1f}s)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sizes = [compile_time_ms, link_time_ms, codegen_time_ms]
labels = ['Compile', 'Link', 'Codegen']
colors = sns.color_palette("muted", len(labels))

wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title(f'Build Time Breakdown (Total: {total_time_ms/1000:.1f}s)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Top Targets by Build Time

Top 30 targets ranked by total build time, broken down into compile, link, and codegen components.

In [ ]:
# Use the pre-computed total_build_time_ms from target_metrics (compile + link + codegen + archive)
# Do NOT overwrite — the consolidation script already computes it correctly.
top_targets = target_metrics.nlargest(30, 'total_build_time_ms')[[
    'cmake_target', 'compile_time_sum_ms', 'link_time_ms', 'total_build_time_ms'
]].copy()

print(f"Top 30 Targets by Total Build Time:")
print(top_targets.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

top_20 = target_metrics.nlargest(20, 'total_build_time_ms')
y_pos = np.arange(len(top_20))

compile_times = top_20['compile_time_sum_ms'].values
link_times = top_20['link_time_ms'].fillna(0).values

ax.barh(y_pos, compile_times, label='Compile', color='steelblue')
ax.barh(y_pos, link_times, left=compile_times, label='Link', color='coral')

ax.set_yticks(y_pos)
ax.set_yticklabels(top_20['cmake_target'].values, fontsize=9)
ax.set_xlabel('Time (ms)', fontsize=11)
ax.set_title('Top 20 Targets by Total Build Time', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## 3. Critical Path Analysis

Build dependency graph analysis to identify the critical path — the longest chain of dependencies that determines minimum wall-clock build time.

In [ ]:
edge_list = ds.edge_list
graph = build_dependency_graph(target_metrics, edge_list)

print(f"Build Dependency Graph:")
print(f"  Targets (nodes): {graph.n_targets}")
print(f"  Dependencies (edges): {graph.n_edges}")

In [ ]:
timing = target_metrics[["cmake_target", "total_build_time_ms"]].copy()
cp = compute_critical_path(graph, timing)
critical_path_nodes = cp.path
critical_path_time_ms = cp.total_time_s * 1000

print(f"\nCritical Path:")
print(f"  Length (wall-clock): {critical_path_time_ms:,.0f} ms ({critical_path_time_ms/1000:.1f}s)")
print(f"  Targets on path: {len(critical_path_nodes)}")
print(f"  Critical path as % of total: {critical_path_time_ms/(total_time_ms)*100:.1f}%")
print(f"\nCritical Path (ordered):")

for i, node in enumerate(critical_path_nodes, 1):
    node_time = graph.graph.nodes[node].get('total_build_time_ms', 0)
    print(f"  {i:2d}. {node:50s} {node_time:10,.0f} ms")

## 4. Build Parallelism

Simulate build times at different core counts to estimate parallelism efficiency.

In [ ]:
core_counts = [1, 2, 4, 8, 16, 32, 64]
simulated_times = []

for cores in core_counts:
    schedule = simulate_build(graph, timing, n_cores=cores)
    wall_clock = schedule["end_ms"].max() if len(schedule) > 0 else 0
    simulated_times.append(wall_clock)
    print(f"  {cores:2d} cores: {wall_clock:10,.0f} ms ({wall_clock/1000:6.1f}s)")

print(f"\nParallelism Efficiency:")
ideal_times = [total_time_ms / c for c in core_counts]
for i, cores in enumerate(core_counts):
    efficiency = ideal_times[i] / simulated_times[i] if simulated_times[i] > 0 else 0
    print(f"  {cores:2d} cores: {efficiency*100:5.1f}% efficiency")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(core_counts, [t/1000 for t in simulated_times], marker='o', linewidth=2, markersize=8, label='Simulated')
ax1.plot(core_counts, [t/1000 for t in ideal_times], marker='s', linewidth=2, markersize=8, label='Ideal (linear scaling)')
ax1.set_xlabel('Number of Cores', fontsize=11)
ax1.set_ylabel('Wall-Clock Time (s)', fontsize=11)
ax1.set_title('Build Time vs Core Count', fontsize=12, fontweight='bold')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.legend()
ax1.grid(True, alpha=0.3)

efficiency = [ideal_times[i] / simulated_times[i] if simulated_times[i] > 0 else 0 for i in range(len(core_counts))]
ax2.plot(core_counts, [e*100 for e in efficiency], marker='o', linewidth=2, markersize=8, color='coral')
ax2.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='Perfect scaling (100%)')
ax2.set_xlabel('Number of Cores', fontsize=11)
ax2.set_ylabel('Parallelism Efficiency (%)', fontsize=11)
ax2.set_title('Parallelism Efficiency', fontsize=12, fontweight='bold')
ax2.set_xscale('log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Pool-aware simulation: demonstrate link_pool throttling effect
pool_cfg = PoolConfig.with_link_pool(link_depth=4)

print("Impact of Link Pool Throttling (depth=4):")
print()

for cores in [8, 16, 32, 64]:
    schedule_no_pool = simulate_build(graph, timing, n_cores=cores)
    wall_no_pool = schedule_no_pool["end_ms"].max() if len(schedule_no_pool) > 0 else 0

    schedule_pooled = simulate_build(graph, timing, n_cores=cores, pool_config=pool_cfg)
    wall_pooled = schedule_pooled["end_ms"].max() if len(schedule_pooled) > 0 else 0

    overhead_pct = (wall_pooled - wall_no_pool) / wall_no_pool * 100 if wall_no_pool > 0 else 0
    print(f"  {cores:2d} cores: {wall_no_pool:10,.0f} ms (no pool) -> {wall_pooled:10,.0f} ms (link_pool=4)  overhead: {overhead_pct:+.1f}%")

## 4b. Simulation Validation

Compare simulated build schedule against observed build_schedule.parquet to quantify simulator accuracy.

In [ ]:
if ds.has_file("build_schedule"):
    observed = ds.build_schedule
    simulated = simulate_build(graph, timing, n_cores=8)
    validation = validate_simulation(simulated, observed)

    print("Simulation Validation (8 cores):")
    for key, value in validation.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.2f}")
        else:
            print(f"  {key}: {value}")
else:
    print("build_schedule.parquet not available — skipping simulation validation.")

## 5. What-If Analysis

Simulate impact of optimizing critical path targets and removing top fan-in edges.

In [ ]:
print("Impact of 50% Reduction in Critical Path Targets:")
print()

reductions = []
for target in critical_path_nodes[:10]:
    result = whatif_reduce_target_time(graph, timing, target, reduction_pct=50)
    saved = result["delta_ms"]
    saved_pct = (saved / critical_path_time_ms) * 100 if critical_path_time_ms > 0 else 0
    reductions.append((target, saved, saved_pct))
    print(f"  {target:50s} → saves {saved:8,.0f} ms ({saved_pct:5.1f}%)")

reductions_df = pd.DataFrame(reductions, columns=['target', 'saved_ms', 'saved_pct'])

In [ ]:
print("\nImpact of Removing Top Fan-In Edges:")
print()

edge_impacts = []
in_degrees = dict(graph.graph.in_degree())
top_fan_in = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:10]

for target, in_degree in top_fan_in:
    predecessors = list(graph.graph.predecessors(target))
    if predecessors:
        pred = predecessors[0]
        try:
            result = whatif_remove_edge(graph, timing, pred, target)
            saved = result["delta_ms"]
            saved_pct = (saved / critical_path_time_ms) * 100 if critical_path_time_ms > 0 else 0
            edge_impacts.append((f"{pred} �� {target}", saved, saved_pct))
            print(f"  Remove {pred:30s} → {target:30s} saves {saved:8,.0f} ms ({saved_pct:5.1f}%)")
        except Exception:
            pass

## 6. Compile vs Link Time

Identify targets that are bottlenecked by linking rather than compilation.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

significant_targets = target_metrics[target_metrics['total_build_time_ms'] > target_metrics['total_build_time_ms'].quantile(0.75)]

ax.scatter(significant_targets['compile_time_sum_ms'], significant_targets['link_time_ms'].fillna(0), 
           s=significant_targets['total_build_time_ms']/100, alpha=0.6, color='steelblue')

for idx, row in significant_targets.nlargest(10, 'total_build_time_ms').iterrows():
    ax.annotate(row['cmake_target'], 
                (row['compile_time_sum_ms'], row['link_time_ms'] if pd.notna(row['link_time_ms']) else 0),
                fontsize=8, alpha=0.7)

ax.set_xlabel('Compile Time (ms)', fontsize=11)
ax.set_ylabel('Link Time (ms)', fontsize=11)
ax.set_title('Compile vs Link Time (bubble size = total time)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

link_heavy = target_metrics[target_metrics['link_time_ms'].fillna(0) > target_metrics['compile_time_sum_ms']]
print(f"\nLink-Heavy Targets (link_time > compile_time):")
if len(link_heavy) > 0:
    print(link_heavy.nlargest(10, 'link_time_ms')[['cmake_target', 'compile_time_sum_ms', 'link_time_ms']].to_string())
else:
    print("  None found")

## 7. Save Intermediates

In [ ]:
# Save critical path analysis for downstream notebooks
ds.save_intermediate("critical_path", cp.target_slack)
print(f"Saved critical_path.parquet ({len(cp.target_slack)} rows, columns: {list(cp.target_slack.columns)})")
print(f"  Critical path length: {cp.total_time_s:.1f}s, parallelism ratio: {cp.parallelism_ratio:.2f}")

## 8. Incremental Build Analysis

Model incremental rebuild times for changes to individual targets, and rank targets by expected daily rebuild cost.

In [ ]:
import networkx as nx

G = graph.graph
target_times = timing.set_index("cmake_target")["total_build_time_ms"].to_dict()

# Incremental rebuild time for modifying each critical path target
print("Incremental Build Time (modifying each critical path target):")
print(f"{'Target':50s} {'Rebuild Time (ms)':>18s} {'Targets Rebuilt':>16s}")
print("-" * 90)

for target in critical_path_nodes[:15]:
    incr_time = simulate_incremental_build(G, [target], target_times, n_cores=8)
    rev = G.reverse()
    rebuild_set = nx.descendants(rev, target) | {target} if target in rev else {target}
    print(f"  {target:50s} {incr_time:14,.0f} ms   {len(rebuild_set):10d}")

# Expected daily rebuild cost — which targets matter most in practice?
print(f"\nTop 20 Targets by Expected Daily Rebuild Cost:")
print(f"{'Target':50s} {'Rebuild Cost (ms)':>18s} {'Daily Cost (ms)':>16s}")
print("-" * 90)

daily_costs = []
for _, row in target_metrics.iterrows():
    target = row["cmake_target"]
    if target not in G:
        continue
    rc = rebuild_cost(G, target, target_metrics)
    dc = expected_daily_cost(G, target, target_metrics, target_metrics)
    daily_costs.append({"target": target, "rebuild_cost_ms": rc, "daily_cost_ms": dc})

daily_costs_df = pd.DataFrame(daily_costs).sort_values("daily_cost_ms", ascending=False)
for _, row in daily_costs_df.head(20).iterrows():
    print(f"  {row['target']:50s} {row['rebuild_cost_ms']:14,.0f} ms   {row['daily_cost_ms']:12,.1f}")

import networkx as nx

## 9. Merge/Split What-If

Simulate merging small adjacent targets and splitting large bottleneck targets.

In [ ]:
# Find critical path targets with high compile time that could benefit from splitting
high_compile = target_metrics[
    target_metrics["cmake_target"].isin(critical_path_nodes)
].nlargest(5, "compile_time_sum_ms")

print("Split What-If (top 5 critical path targets by compile time):")
for _, row in high_compile.iterrows():
    target = row["cmake_target"]
    file_count = int(row.get("file_count", 0) or 0)
    if file_count >= 2:
        # Simulate splitting into 2 equal partitions
        half = file_count // 2
        result = simulate_split(G, target, [["group_a"] * half, ["group_b"] * (file_count - half)], target_metrics)
        print(f"\n  {target} ({file_count} files):")
        print(f"    Before: {result['before'].get('total_build_time_ms', 0):,.0f} ms")
        print(f"    Split into {len(result['partitions'])} partitions")
        for note in result["notes"]:
            print(f"    Note: {note}")

# Find pairs of small adjacent targets that could be merged
small_targets = target_metrics[
    (target_metrics["total_build_time_ms"] < target_metrics["total_build_time_ms"].quantile(0.25)) &
    (target_metrics["cmake_target"].isin(G.nodes()))
]["cmake_target"].tolist()

merge_candidates = []
for t in small_targets[:50]:
    for dep in G.successors(t):
        if dep in small_targets:
            result = simulate_merge(G, [t, dep], target_metrics)
            if result["savings_ms"] > 0:
                merge_candidates.append({"pair": f"{t} + {dep}", **result})

if merge_candidates:
    merge_df = pd.DataFrame(merge_candidates).sort_values("savings_ms", ascending=False)
    print(f"\nMerge What-If (top 10 candidates by savings):")
    for _, row in merge_df.head(10).iterrows():
        print(f"  {row['pair']:60s} saves {row['savings_ms']:6,.0f} ms")
else:
    print("\nNo profitable merge candidates found among small targets.")

## 10. Historical Build Replay

Replay git history through the incremental build model to estimate per-commit developer wait times.

In [ ]:
if ds.has_file("git_commit_log") and ds.has_file("file_metrics"):
    git_log = ds.git_commit_log
    file_metrics = ds.file_metrics

    from buildanalysis.git import compute_file_to_target_map
    file_to_target_map = compute_file_to_target_map(file_metrics).to_dict()

    replay_df = replay_git_history(G, git_log, file_to_target_map, target_times, n_cores=8)

    if len(replay_df) > 0:
        print(f"Git History Build Replay ({len(replay_df)} commits):")
        print(f"  Mean incremental build time: {replay_df['build_time_ms'].mean():,.0f} ms")
        print(f"  Median: {replay_df['build_time_ms'].median():,.0f} ms")
        print(f"  P90: {replay_df['build_time_ms'].quantile(0.9):,.0f} ms")
        print(f"  P99: {replay_df['build_time_ms'].quantile(0.99):,.0f} ms")
        print(f"  Max: {replay_df['build_time_ms'].max():,.0f} ms")
        print(f"  Commits with zero rebuild: {(replay_df['build_time_ms'] == 0).sum()}")

        fig, ax = plt.subplots(figsize=(12, 5))
        nonzero = replay_df[replay_df["build_time_ms"] > 0]["build_time_ms"]
        ax.hist(nonzero / 1000, bins=50, color="steelblue", alpha=0.7, edgecolor="black")
        ax.axvline(nonzero.median() / 1000, color="orange", linestyle="--", label=f"Median: {nonzero.median()/1000:.1f}s")
        ax.axvline(nonzero.quantile(0.9) / 1000, color="red", linestyle="--", label=f"P90: {nonzero.quantile(0.9)/1000:.1f}s")
        ax.set_xlabel("Incremental Build Time (seconds)")
        ax.set_ylabel("Number of Commits")
        ax.set_title("Distribution of Per-Commit Incremental Build Times")
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("git_commit_log or file_metrics not available — skipping history replay.")